In [ ]:
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()

In [ ]:
%pip install torch torch_geometric
dbutils.library.restartPython()

In [ ]:
from datetime import date, timedelta

import pyspark.sql.functions as F
from pyspark.sql import Column
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.window import Window

def get_customer(spark: SparkSession, customer_shortname: str) -> DataFrame:
    """Résout un client vers son publisherId Artemis.

    Args:
        spark: Session Spark active.
        customer_shortname: shortName du client dans t2s_gold_customer.

    Returns:
        DataFrame avec les colonnes customer_shortname, customerId, db_name,
        publisherId. Un client sans publisher Artemis correspondant (namespace
        artemis-prod) est absent du résultat (inner join).
    """
    customers = (
        spark.table("mirakl_ai.ds_etl_prod.t2s_gold_customer")
        .where(F.col("shortName") == customer_shortname)
        .select("shortName", "publicId", F.col("databaseName").alias("db_name"))
    )

    publishers = (
        spark.table("mirakl_data_platform.prod_data_platform_silver.artemis_insights_customer").alias("I")
        .join(
            spark.table("mirakl_data_platform.prod_data_platform_silver.artemis_publisher").alias("P"),
            F.col("P.insights_customer_id") == F.col("I.id"),
            "left",
        )
        .where(F.col("I.__k8s_namespace") == "artemis-prod")
        .select(F.col("P.id").alias("publisherId"), F.col("I.public_id").alias("public_id"))
    )

    return (
        customers.join(publishers, customers.publicId == publishers.public_id, "inner")
        .select(
            F.col("shortName").alias("customer_shortname"),
            F.col("publicId").alias("customerId"),
            "db_name",
            "publisherId"
        )
    )

def get_products(spark: SparkSession, db_names: list[str]) -> DataFrame:
    """Charge le catalogue des produits maîtres actifs.

    Args:
        spark: Session Spark active.
        db_names: Bases clients dont on charge le catalogue.

    Returns:
        DataFrame avec les colonnes internalId (bigint), fwProductId, name,
        imageUrl. Une ligne par produit maître (parentId = -1), actif et
        diffusé (isActive, isOkInStream) ; les déclinaisons et les produits
        désactivés ou hors flux sont absents.
    """
    return (
        spark.table("mirakl_ai.ds_etl_prod.t2s_mongo_product_0_current")
        .filter(
            F.col("db_name").isin(db_names)
            & F.col("isActive")
            & F.col("isOkInStream")
            & (F.col("parentId") == -1)
        )
        .select(
            F.col("internalId").cast("bigint").alias("internalId"),
            "fwProductId",
            "name",
            "imageUrl"
        )
    )

def get_embeddings(spark: SparkSession, db_names: list[str]) -> DataFrame:
    """Charge les embeddings des produits pour une version de modèle fixée.

    Args:
        spark: Session Spark active.
        db_names: Bases clients dont on charge les embeddings.

    Returns:
        DataFrame avec les colonnes internalId, embeddings, last_update_date.
        Une ligne par produit ayant un embedding calculé par le modèle
        last_model_id fixé dans le code ; les produits embeddés par une
        autre version sont absents.
    """
    return (
        spark.table("mirakl_ai.ds_artemis_prod.product_embeddings")
        .filter(
            F.col("db_name").isin(db_names)
            & (F.col("last_model_id") == "06bbe5791147457aa395960da021c05c")
        )
        .select(
            "internalId",
            "embeddings",
            "last_update_date"
        )
    )

def get_views(spark: SparkSession, customer_ids: list[str], start_date: date, end_date: date) -> DataFrame:
    """Charge les vues produit brutes sur une fenêtre de dates, réconciliées par utilisateur.

    Args:
        spark: Session Spark active.
        customer_ids: Identifiants clients dont on charge les vues.
        start_date: Début de la fenêtre (inclus).
        end_date: Fin de la fenêtre — les bornes sont converties en
            timestamp à minuit, donc la journée de end_date est exclue.

    Returns:
        DataFrame avec les colonnes emitted, internalId, userId. Une ligne
        par vue produit, restreinte aux visiteurs identifiés
        (user.internalId non nul) et résolus en produit maître
        (masterProductInternalId non nul) ; userId est l'identité
        réconciliée via t2s_merged_user quand elle existe.
    """
    df_merged_user = (
        spark.table("mirakl_data_platform.prod_data_platform_silver.t2s_merged_user")
        .filter(F.col("customerId").isin(customer_ids))
        .select(
            F.col("masterId").alias("userMasterId"),
            F.col("sourceId").alias("userId"),
        )
    )

    return (
        spark.table("mirakl_data_platform.prod_data_platform_silver.t2s_tracking_product_display_page_event_fct")
        .filter(
            F.col("customerId").isin(customer_ids)
            & F.col("emitted").between(start_date, end_date)
            & F.col("user.internalId").isNotNull()
            & F.col("masterProductInternalId").isNotNull()
        )
        .withColumnRenamed("masterProductInternalId", "internalId")
        .withColumn("userId", F.col("user.internalId"))
        .join(df_merged_user, on="userId", how="left")
        .withColumn("userId", F.coalesce(F.col("userMasterId"), F.col("userId")))
        .select(
            "emitted", # timestamp of the event
            "internalId", # product internalId
            "userId", # only used for aggregation, doesn't carry meaning
        )
    )

def sessionize_views(df_views: DataFrame, session_timeout: int = 1800) -> DataFrame:
    """Découpe les vues d'un utilisateur en sessions par expiration d'inactivité.

    Args:
        df_views: Vues par utilisateur (colonnes userId, internalId, emitted).
        session_timeout: Durée d'inactivité en secondes au-delà de laquelle
            une nouvelle session démarre.

    Returns:
        DataFrame avec les colonnes userId, session_id, internalId, emitted,
        session_start. Une ligne par vue, enrichie de son identifiant de
        session et de l'horodatage de début de session ; aucune vue n'est
        retirée.
    """
    user_window = Window.partitionBy("userId").orderBy("emitted", "internalId")

    df = df_views.withColumn(
        "seconds_since_last", 
        (F.col("emitted") - F.lag("emitted").over(user_window)).cast("long")
    )

    df = df.withColumn(
        "is_new_session",
        F.when(
            F.col("seconds_since_last").isNull() | (F.col("seconds_since_last") > session_timeout),
            F.lit(1)
        ).otherwise(F.lit(0))
    )

    df = df.withColumn(
        "session_id",
        F.sum("is_new_session").over(user_window.rowsBetween(Window.unboundedPreceding, Window.currentRow))
    )

    session_window = Window.partitionBy("userId", "session_id")
    df = df.withColumn(
        "session_start",
        F.min("emitted").over(session_window)
    )

    return df.select("userId", "session_id", "internalId", "emitted", "session_start")

def split_sessions(df_sessionized: DataFrame, cutoff_train: date, cutoff_val: date) -> DataFrame:
    """Assigne chaque session à train/val/test selon deux coupures de date.

    Args:
        df_sessionized: Vues sessionisées (colonnes userId, session_id,
            session_start, ...).
        cutoff_train: Date avant laquelle une session part en train.
        cutoff_val: Date avant laquelle (et après cutoff_train) une session
            part en val ; au-delà, elle part en test.

    Returns:
        DataFrame identique à df_sessionized, enrichi d'une colonne split
        (train/val/test). La coupure se fait sur session_start : toutes les
        vues d'une même session tombent du même côté, même si la session
        déborde sur une autre période.
    """
    df_sessions = df_sessionized.select("userId", "session_id", "session_start").distinct()

    df_sessions = df_sessions.withColumn(
        "split",
        F.when(F.col("session_start") < F.lit(cutoff_train), F.lit("train"))
        .when(F.col("session_start") < F.lit(cutoff_val), F.lit("val"))
        .otherwise(F.lit("test"))
    )

    return df_sessionized.join(
        df_sessions.select("userId", "session_id", "split"),
        on=["userId", "session_id"],
        how="inner",
    )

USE_SESSION_WEIGHT_NORM = False

def build_co_view_edges(df_sessionized: DataFrame) -> DataFrame:
    """Construit les arêtes de co-vue à partir des vues sessionisées.

    Args:
        df_sessionized: Vues sessionisées (colonnes userId, session_id,
            internalId).

    Returns:
        DataFrame avec les colonnes product_A, product_B, weight, raw_count.
        Une ligne par paire de produits distincts vus dans au moins une même
        session (product_A < product_B) ; weight compte les sessions
        distinctes contenant la paire (ou une somme pondérée par la taille
        de session si USE_SESSION_WEIGHT_NORM est activé), et raw_count le
        nombre brut d'occurrences. Une session à un seul produit distinct ne
        produit aucune arête.
    """
    df_dedup = df_sessionized.select("userId", "session_id", "internalId").distinct()

    session_sizes = (
        df_dedup
        .groupBy("userId", "session_id")
        .agg(F.countDistinct("internalId").alias("session_size"))
    )

    df_dedup = df_dedup.join(session_sizes, on=["userId", "session_id"], how="inner")

    df_pairs = (
        df_dedup.alias("L")
        .join(
            df_dedup.alias("R"),
            on=(
                (F.col("L.userId") == F.col("R.userId"))
                & (F.col("L.session_id") == F.col("R.session_id"))
                & (F.col("L.internalId") < F.col("R.internalId"))
            ),
            how="inner"
        )
        .select(
            F.col("L.internalId").alias("product_A"),
            F.col("R.internalId").alias("product_B"),
            (
                (F.lit(1.0) / (F.col("L.session_size") - F.lit(1)))
                if USE_SESSION_WEIGHT_NORM
                else F.lit(1.0)
            ).alias("pair_weight"),
        )
    )

    return df_pairs.groupBy("product_A", "product_B").agg(
        F.sum("pair_weight").alias("weight"),
        F.count("*").alias("raw_count"),
    )

def remove_train_edges(df_edges: DataFrame, df_train_edges: DataFrame) -> DataFrame:
    """Retire de df_edges les paires déjà présentes dans df_train_edges.

    Args:
        df_edges: Arêtes candidates (val ou test), colonnes product_A,
            product_B.
        df_train_edges: Arêtes d'entraînement à exclure.

    Returns:
        DataFrame identique à df_edges, restreint aux paires absentes de
        df_train_edges. Les paires communes à val et test entre elles ne
        sont pas dédupliquées : une même paire peut se retrouver dans les
        deux.
    """
    return df_edges.join(
        df_train_edges.select("product_A", "product_B"),
        on=["product_A", "product_B"],
        how="left_anti"
    )

In [ ]:
df_customer = get_customer(spark, customer_shortname="maisons-du-monde")
customer_rows = df_customer.collect()
customer_ids = [row["customerId"] for row in customer_rows]
publisher_ids = [row["publisherId"] for row in customer_rows]
db_names = [row["db_name"] for row in customer_rows]

df_products = get_products(spark, db_names)
df_embeddings = get_embeddings(spark, db_names)

start_date = date(2026, 1, 1)
end_date = date(2026, 5, 1)
cutoff_val = end_date - timedelta(days=15)
cutoff_train = end_date - timedelta(days=30)
assert start_date < cutoff_train < cutoff_val < end_date

df_views = get_views(spark, customer_ids, start_date, end_date)

In [ ]:
df_sessionized = sessionize_views(df_views.join(df_products, on="internalId", how="inner"))

df_split = split_sessions(df_sessionized, cutoff_train=cutoff_train, cutoff_val=cutoff_val)
df_split = df_split.cache()

def build(df_subset, df_train_edges=None):
    """Construit les arêtes de co-vue d'un sous-ensemble, en retirant celles du train si fourni.

    Args:
        df_subset: Vues sessionisées restreintes à un split (train, val ou
            test).
        df_train_edges: Arêtes d'entraînement à exclure des arêtes
            produites. Laissé à None pour le split train lui-même.

    Returns:
        DataFrame d'arêtes de co-vue (colonnes product_A, product_B,
        weight, raw_count) pour df_subset, débarrassé des paires déjà en
        train si df_train_edges est fourni.
    """
    edges = build_co_view_edges(df_subset)
    if df_train_edges is not None:
        edges = remove_train_edges(edges, df_train_edges)
    return edges

df_train_edges = build(df_split.filter(F.col("split") == "train"))
df_val_edges = build(df_split.filter(F.col("split") == "val"), df_train_edges)
df_test_edges = build(df_split.filter(F.col("split") == "test"), df_train_edges)

def count_nodes(df_edges: DataFrame) -> int:
    """Compte le nombre de produits distincts apparaissant dans un jeu d'arêtes.

    Args:
        df_edges: Arêtes de co-vue (colonnes product_A, product_B).

    Returns:
        Nombre de valeurs distinctes réunies sur product_A et product_B.
    """
    return (
        df_edges.select(F.col("product_A").alias("node"))
        .union(df_edges.select(F.col("product_B").alias("node")))
        .distinct()
        .count()
    )

all_edges = df_train_edges.union(df_val_edges).union(df_test_edges)
total_nodes = count_nodes(all_edges)

print(f"Total nodes: {total_nodes}")
print(f"Train  — edges: {df_train_edges.count()}")
print(f"Val    — edges: {df_val_edges.count()}")
print(f"Test   — edges: {df_test_edges.count()}")

In [ ]:
import pandas

df_graph_nodes = (
    df_train_edges.select(F.col("product_A").alias("internalId"))
    .union(df_train_edges.select(F.col("product_B").alias("internalId")))
    .union(df_val_edges.select(F.col("product_A").alias("internalId")))
    .union(df_val_edges.select(F.col("product_B").alias("internalId")))
    .union(df_test_edges.select(F.col("product_A").alias("internalId")))
    .union(df_test_edges.select(F.col("product_B").alias("internalId")))
    .distinct()
)

df_train_edges_pandas = df_train_edges.toPandas()
df_val_edges_pandas = df_val_edges.toPandas()
df_test_edges_pandas = df_test_edges.toPandas()

emb_pandas = df_embeddings.join(df_graph_nodes, on="internalId", how="inner").toPandas()

In [ ]:
import pandas as pd
import numpy as np

def build_node_mapping(*edge_dfs: pd.DataFrame) -> dict[int, int]:
    """Construit une correspondance internalId -> index contigu (0..N-1).

    Args:
        *edge_dfs: Un ou plusieurs DataFrames d'arêtes (typiquement train,
            val, test), chacun avec les colonnes product_A, product_B.

    Returns:
        Dictionnaire internalId -> index. Le jeu de nœuds est l'union de
        product_A et product_B sur tous les DataFrames fournis ; les index
        sont attribués dans l'ordre croissant des internalId, donc
        déterministes pour un jeu de nœuds donné.
    """

    all_nodes = pd.concat(
        [df["product_A"] for df in edge_dfs] + [df["product_B"] for df in edge_dfs]
    ).unique()

    all_nodes.sort()
    node2idx = {node_id: idx for idx, node_id in enumerate(all_nodes)} 

    return node2idx

def remap_edges(df: pd.DataFrame, node2idx: dict[int, int]) -> pd.DataFrame:
    """Remplace les internalId par leurs index contigus dans un DataFrame d'arêtes.

    Args:
        df: DataFrame d'arêtes (colonnes product_A, product_B) avec des
            internalId bruts.
        node2idx: Correspondance internalId -> index, typiquement
            construite par build_node_mapping sur ce même DataFrame (entre
            autres).

    Returns:
        Copie de df où product_A et product_B contiennent les index
        mappés. Un internalId absent de node2idx devient NaN
        silencieusement (pas d'erreur, pas de ligne retirée) : cette
        fonction suppose que node2idx couvre déjà tous les internalId de
        df.
    """
    df = df.copy()
    df["product_A"] = df["product_A"].map(node2idx)
    df["product_B"] = df["product_B"].map(node2idx)
    return df

In [ ]:
# Build node mapping across all 3 splits
node2idx = build_node_mapping(df_train_edges_pandas, df_val_edges_pandas, df_test_edges_pandas)

print(f"Number of nodes: {len(node2idx)}")

# Apply mapping to remap internalIds to contiguous indices
train_remapped = remap_edges(df_train_edges_pandas, node2idx)
val_remapped = remap_edges(df_val_edges_pandas, node2idx)
test_remapped = remap_edges(df_test_edges_pandas, node2idx)

In [ ]:
import torch

embedding_dim = len(emb_pandas["embeddings"].iloc[0])

node_features = np.zeros((len(node2idx), embedding_dim), dtype=np.float32)

emb_matrix = np.vstack(emb_pandas["embeddings"].values).astype(np.float32)
positions = emb_pandas["internalId"].map(node2idx).values.astype(int)
node_features[positions] = emb_matrix

x = torch.tensor(node_features, dtype=torch.float)

print(f"x shape: {x.shape}")
print(f"Nodes without embedding: {(node_features == 0).all(axis=1).sum()}")

In [ ]:
def edges_to_tensors(df: pd.DataFrame, symmetric: bool = True) -> tuple[torch.Tensor, torch.Tensor]:
    """Convertit un DataFrame d'arêtes remappées en tensors compatibles PyG.

    Args:
        df: DataFrame d'arêtes remappées (colonnes product_A, product_B,
            weight), avec des index de nœuds contigus.
        symmetric: Si True, ajoute l'arête inverse (B->A) pour chaque
            arête (A->B), pour un graphe non dirigé.

    Returns:
        Tuple (edge_index, edge_attr). edge_index a la forme [2, E] (ou
        [2, 2E] si symmetric), edge_attr la forme [E] (ou [2E]) avec les
        poids dupliqués à l'identique dans les deux sens.
    """
    src = df["product_A"].values
    dst = df["product_B"].values
    weight = df["weight"].values

    if symmetric:
        # Stack both directions: A→B and B→A
        edge_index = np.stack([
            np.concatenate([src, dst]),
            np.concatenate([dst, src]),
        ])
        edge_attr = np.concatenate([weight, weight])
    else:
        edge_index = np.stack([src, dst])
        edge_attr = weight

    return (
        torch.tensor(edge_index, dtype=torch.long),
        torch.tensor(edge_attr, dtype=torch.float),
    )

# Convert remapped DataFrames to PyG tensors (symmetric for undirected graph)
train_edge_index, train_edge_attr = edges_to_tensors(train_remapped, symmetric=True)
val_edge_index, val_edge_attr = edges_to_tensors(val_remapped, symmetric=True)
test_edge_index, test_edge_attr = edges_to_tensors(test_remapped, symmetric=True)

num_nodes = len(node2idx)

print(f"Number of nodes: {num_nodes}")
print(f"Train edges (symmetric): {train_edge_index.shape[1]}")  
print(f"Val edges (symmetric): {val_edge_index.shape[1]}")
print(f"Test edges (symmetric): {test_edge_index.shape[1]}")
print(f"Train edge_index shape: {train_edge_index.shape}")     
print(f"Train edge_attr shape: {train_edge_attr.shape}") 

In [ ]:
def recommended_products_for_t2s_user(
    spark: SparkSession,
    customer_ids: list[str],
    publisher_ids: list[str],
    start_date: date,
    end_date: date,
) -> DataFrame:
    """Associe les logs publicitaires t2s aux vues produit qui les ont suivis.

    Args:
        spark: Session Spark active.
        customer_ids: Identifiants clients concernés.
        publisher_ids: Identifiants publisher Artemis concernés.
        start_date: Début de la fenêtre (inclus).
        end_date: Fin de la fenêtre (bornes à minuit, comme pour get_views).

    Returns:
        DataFrame avec une ligne par vue produit appariée à un log
        publicitaire du même cookie/produit/client, dans une fenêtre de
        ±3 secondes, restreint aux pages de type PRODUCT. Colonnes :
        internalId, emitted, userId (réconcilié via t2s_merged_user),
        customerId, executionId, logInstant, pageType, searchTerm,
        userOrganizeRank, relevantProducts,
        sponsoredProductPlacementExecutions, productsReturned.
    """

    df_merged_user = (
        spark.table("mirakl_data_platform.prod_data_platform_silver.t2s_merged_user")
        .filter(F.col("customerId").isin(customer_ids))
        .select(
            F.col("masterId").alias("userMasterId"),
            F.col("sourceId").alias("userId"),
        )
    )

    df_t2s_recommended_products = (
        spark.table("mirakl_data_platform.prod_data_platform_silver.ads_adlog_fct").alias("adlog")
        .filter(
            (F.col("adlog.__k8s_namespace") == "target2sell-prod")
            & F.col("adlog.publisherId").isin(publisher_ids)
            & F.col("adlog.__log_date").between(start_date, end_date)
            & (F.col("adlog.pageType") == "PRODUCT")
        )
        .join(
            spark.table("mirakl_data_platform.prod_data_platform_silver.t2s_tracking_product_display_page_event_fct").alias("t2s")
            .filter(
                F.col("t2s.customerId").isin(customer_ids)
                & F.col("t2s.emitted").between(start_date, end_date)
                & F.col("t2s.masterProductInternalId").isNotNull()
                & F.col("t2s.user.internalId").isNotNull()
            ),
            (F.col("adlog.productId") == F.col("t2s.fwProductId"))
            & (F.col("adlog.userCookie") == F.col("t2s.user.cookie"))
            & (F.col("adlog.customerId") == F.col("t2s.customerId"))
            & (F.col("t2s.emitted").between(
                    F.col("adlog.logInstant") - F.expr("INTERVAL 3 SECONDS"),
                    F.col("adlog.logInstant") + F.expr("INTERVAL 3 SECONDS"),
                )),
            how="inner",
        )
        .withColumn("userId", F.col("t2s.user.internalId"))
        .join(df_merged_user, on="userId", how="left")
        .withColumn("userId", F.coalesce(F.col("userMasterId"), F.col("userId")))
        .select(
            F.col("t2s.masterProductInternalId").alias("internalId"),
            F.col("t2s.emitted").alias("emitted"),
            F.col("userId"),
            F.col("t2s.customerId").alias("customerId"),
            F.col("adlog.executionId"),
            F.col("adlog.logInstant"),
            F.col("adlog.pageType"),
            F.col("adlog.searchTerm"),
            F.col("adlog.userOrganizeRank"),
            F.col("adlog.relevantProducts"),
            F.col("adlog.sponsoredProductPlacementExecutions"),
            F.col("adlog.productsReturned"),
        )
    )
    return df_t2s_recommended_products


In [ ]:
df_recommended_products_t2s_with_sessions_indexes = (
    df_split
    .join(
        recommended_products_for_t2s_user(spark, customer_ids, publisher_ids, start_date, end_date),
        on=["internalId", "emitted", "userId"],
        how="inner",
    )
)

df_recommended_products_t2s_with_sessions_indexes = df_recommended_products_t2s_with_sessions_indexes.cache()

In [ ]:
def positive_negative_edge_construction(df_recommended: DataFrame, df_split: DataFrame, df_train_edges: DataFrame, node2idx: dict[int, int], split: str) -> tuple:
    """Construit les paires positives et négatives d'un split pour l'entraînement du modèle.

    Args:
        df_recommended: Sortie de recommended_products_for_t2s_user jointe
            aux sessions (colonnes split, userId, session_id, internalId,
            executionId, sponsoredProductPlacementExecutions).
        df_split: Vues sessionisées avec split (colonnes userId, session_id,
            internalId).
        df_train_edges: Arêtes de co-vue d'entraînement, utilisées pour
            écarter les faux négatifs.
        node2idx: Correspondance internalId -> index de nœud.
        split: Split à traiter ("train", "val" ou "test").

    Returns:
        Tuple (pos_edge_index, neg_edge_index, production_mrr_matched,
        production_mrr_coverage, exec2code, code2exec). pos/neg_edge_index
        sont des tensors [3, N] (exec_code, trigger, candidat) ; les deux
        MRR sont calculés sur la population complète des exécutions du
        split, avant tout filtrage sur node2idx. exec2code couvre toutes
        les exécutions du split, indépendamment du label.
    """

    df_recommended_products_t2s_with_sessions = (
        df_recommended
        .filter(F.col("split") == split)
        .select(
            F.col("userId"),
            F.col("session_id"),
            F.col("internalId").alias("internalId_trigger"),
            F.col("executionId"),
            F.col("sponsoredProductPlacementExecutions"),
        )
        .join(
            df_split.filter(F.col("split") == split).select(
                F.col("userId"),
                F.col("session_id"),
                F.col("internalId").alias("internalId_session"),
            ),
            on=["userId", "session_id"],
            how="inner"
        )
    )

    # rank_candidate = position d'affichage réelle (1-based) ; trier par relevanceScore
    # nécessiterait un explode + row_number() à la place de posexplode.
    df_recommended_exploded = (
        df_recommended_products_t2s_with_sessions
        .withColumn("placement", F.explode("sponsoredProductPlacementExecutions"))
        .withColumn(
            "top_relevant_products",
            F.slice(F.col("placement.relevantProducts"), 1, 12),
        )
        .select("*", F.posexplode("top_relevant_products").alias("rank_candidate", "candidate"))
        .select(
            F.col("userId"),
            F.col("session_id"),
            F.col("executionId"),
            F.col("internalId_trigger"),
            F.col("internalId_session"),
            F.col("candidate.internalId").alias("internalId_candidate"),
            (F.col("rank_candidate") + F.lit(1)).alias("rank_candidate"),
        )
    ).cache()

    positive_candidates = (
        df_recommended_exploded
        .filter(F.col("internalId_session") == F.col("internalId_candidate"))
        .select(
            F.col("executionId"),
            F.col("internalId_trigger").alias("product_A"),
            F.col("internalId_candidate").alias("product_B"),
        )
        .distinct()
    ).cache()

    # Production MRR : métrique statique de qualité de classement de l'algorithme de prod lui-même.
    matched_rank = (
        df_recommended_exploded
        .filter(F.col("internalId_session") == F.col("internalId_candidate"))
        .groupBy("executionId")
        .agg(F.min("rank_candidate").alias("best_rank"))
        .withColumn("reciprocal_rank", F.lit(1.0) / F.col("best_rank"))
    )

    total_executions = df_recommended_exploded.select("executionId").distinct().count()

    # Une seule agrégation pour éviter de recalculer deux fois le shuffle filter+groupBy.
    match_stats = matched_rank.agg(
        F.count("*").alias("matched_executions"),
        F.sum("reciprocal_rank").alias("sum_reciprocal_rank"),
    ).collect()[0]
    matched_executions = match_stats["matched_executions"]
    sum_reciprocal_rank = match_stats["sum_reciprocal_rank"] or 0.0

    # Coverage-adjusted MRR : les exécutions sans candidat comptent 0, dénominateur = toutes les exécutions.
    production_mrr_coverage = sum_reciprocal_rank / total_executions if total_executions > 0 else 0.0

    # Matched-only MRR : moyenne uniquement sur les exécutions avec un positif connu, seule variante
    # comparable à MRR_trigger (base_metrics.py::MRR_trigger._mrr).
    production_mrr_matched = sum_reciprocal_rank / matched_executions if matched_executions > 0 else 0.0
    coverage = matched_executions / total_executions if total_executions > 0 else 0.0

    print(
        f"[{split}] Production MRR (matched-only, comparable to MRR_trigger): {production_mrr_matched:.4f} | "
        f"Production MRR (coverage-adjusted): {production_mrr_coverage:.4f} | "
        f"{matched_executions}/{total_executions} executions matched (coverage={coverage:.2%})"
    )

    df_all_candidates = (
        df_recommended_exploded
        .select(
            F.col("executionId"),
            F.col("internalId_trigger").alias("product_A"),
            F.col("internalId_candidate").alias("product_B"),
            F.col("rank_candidate")
        )
        .groupBy("executionId", "product_A", "product_B")
        .agg(F.min("rank_candidate").alias("rank_candidate"))
    )

    df_existing_pairs = (
        df_train_edges.select("product_A", "product_B")
        .union(
            df_train_edges.select(
                F.col("product_B").alias("product_A"),
                F.col("product_A").alias("product_B"),
            )
        )
        .distinct()
    )

    # Left join pour distinguer les vrais négatifs (absents du graphe) des faux négatifs (déjà une arête).
    df_negative_labeled = (
        df_all_candidates
        .join(
            df_existing_pairs.withColumn("is_existing_edge", F.lit(True)),
            on=["product_A", "product_B"],
            how="left"
        )
        .fillna(False, subset=["is_existing_edge"])
    )

    # Cached : réutilisé plus bas pour exec_only_negatives et la jointure left_semi finale.
    negative_candidates = (
        df_negative_labeled
        .filter(~F.col("is_existing_edge"))
        .select(
            F.col("executionId"),
            F.col("product_A"),
            F.col("product_B"),
            F.col("rank_candidate"),
        )
        .join(
            positive_candidates.select("executionId", "product_A", "product_B"),
            on=["executionId", "product_A", "product_B"],
            how="left_anti"
        )
    ).cache()

    deepest_pos_rank = (
        df_recommended_exploded
        .filter(F.col("internalId_session") == F.col("internalId_candidate"))
        .groupBy("executionId")
        .agg(F.max("rank_candidate").alias("deepest_pos_rank"))
    )

    # Réduit le volume de négatifs : les exécutions sans aucun positif reçoivent fraction=0.0
    # (exclues des négatifs d'entraînement).
    exec_with_positives = positive_candidates.select("executionId").distinct()

    negative_candidates_matched = (
        negative_candidates
        .join(F.broadcast(deepest_pos_rank), on="executionId", how="inner")
        .filter(F.col("rank_candidate") <= F.col("deepest_pos_rank"))
        .select("executionId", "product_A", "product_B")
    )

    exec_only_negatives_full = (
        negative_candidates.select("executionId").distinct()
        .join(exec_with_positives, on="executionId", how="left_anti")
    )

    exec_only_negatives = exec_only_negatives_full.sample(fraction=0.0, seed=42)

    negative_candidates_pure_negative = (
        negative_candidates
        .join(F.broadcast(exec_only_negatives), on="executionId", how="left_semi")
        .select("executionId", "product_A", "product_B")
    )

    negative_candidates_sampled = negative_candidates_matched.union(negative_candidates_pure_negative)

    # Conversion en pandas séparée — pas de cross-join, l'échantillonnage se fait dans le dataloader.
    positive_pandas = positive_candidates.toPandas()
    negative_pandas = negative_candidates_sampled.toPandas()

    # exec2code couvre toutes les exécutions du split, indépendamment du label, pour que les
    # jointures inner en aval (sessions_raw_val_prototype, prod_results_val_prototype) ne perdent aucun trigger.
    all_exec_ids = pd.concat([
        positive_pandas["executionId"],
        negative_pandas["executionId"],
        df_recommended.filter(F.col("split") == split).select("executionId").distinct().toPandas()["executionId"],
    ]).unique()
    # Ordre déterministe : sinon exec_code dépend de l'ordre de collecte non déterministe de Spark.
    all_exec_ids.sort()
    exec2code = {eid: i for i, eid in enumerate(all_exec_ids)}
    # all_exec_ids est déjà ordonné par code (index i <-> code i), donc c'est déjà code2exec.
    code2exec = all_exec_ids.tolist()

    positive_pandas["exec_code"] = positive_pandas["executionId"].map(exec2code)
    negative_pandas["exec_code"] = negative_pandas["executionId"].map(exec2code)

    for col in ["product_A", "product_B"]:
        positive_pandas[col] = positive_pandas[col].map(node2idx)
        negative_pandas[col] = negative_pandas[col].map(node2idx)

    positive_pandas = positive_pandas.dropna(subset=["product_A", "product_B", "exec_code"])
    negative_pandas = negative_pandas.dropna(subset=["product_A", "product_B", "exec_code"])

    # Tensors 3D [3, N] : [exec_code, trigger, product].
    pos_edge_index = torch.tensor(
        positive_pandas[["exec_code", "product_A", "product_B"]].values.T,
        dtype=torch.long
    )

    neg_edge_index = torch.tensor(
        negative_pandas[["exec_code", "product_A", "product_B"]].values.T,
        dtype=torch.long
    )

    print(f"pos_edge_index shape: {pos_edge_index.shape}")
    print(f"neg_edge_index shape: {neg_edge_index.shape}")

    # df_positives_rank / negative_candidates / df_all_candidates ne sont pas retournés : les
    # tables prod sont reconstruites indépendamment en aval.
    return (
        pos_edge_index,
        neg_edge_index,
        production_mrr_matched,
        production_mrr_coverage,
        exec2code,
        code2exec,
    )


train_pos_edge_index, train_neg_edge_index, train_production_mrr, train_production_mrr_coverage, train_exec2code, train_code2exec = positive_negative_edge_construction(df_recommended_products_t2s_with_sessions_indexes, df_split, df_train_edges, node2idx, "train")
val_pos_edge_index, val_neg_edge_index, val_production_mrr, val_production_mrr_coverage, val_exec2code, val_code2exec = positive_negative_edge_construction(df_recommended_products_t2s_with_sessions_indexes, df_split, df_train_edges, node2idx, "val")
test_pos_edge_index, test_neg_edge_index, test_production_mrr, test_production_mrr_coverage, test_exec2code, test_code2exec = positive_negative_edge_construction(df_recommended_products_t2s_with_sessions_indexes, df_split, df_train_edges, node2idx, "test")

print(f"Train pos_edge_index shape: {train_pos_edge_index.shape}")
print(f"Train neg_edge_index shape: {train_neg_edge_index.shape}")

print(f"Val pos_edge_index shape: {val_pos_edge_index.shape}")
print(f"Val neg_edge_index shape: {val_neg_edge_index.shape}")

print(f"Test pos_edge_index shape: {test_pos_edge_index.shape}")
print(f"Test neg_edge_index shape: {test_neg_edge_index.shape}")

print(f"Train production MRR (matched-only, comparable to MRR_trigger): {train_production_mrr:.4f} | coverage-adjusted: {train_production_mrr_coverage:.4f}")
print(f"Val production MRR (matched-only, comparable to MRR_trigger): {val_production_mrr:.4f} | coverage-adjusted: {val_production_mrr_coverage:.4f}")
print(f"Test production MRR (matched-only, comparable to MRR_trigger): {test_production_mrr:.4f} | coverage-adjusted: {test_production_mrr_coverage:.4f}")


Prototype — toutes les sessions sont traitées de la même façon : plus de séparation entre sessions avec positifs et sessions full-negatives.

`exec_code` couvre désormais **toutes** les exécutions du split, donc aucun trigger n'est perdu par les joins.

In [ ]:
val_triggers = (
    df_recommended_products_t2s_with_sessions_indexes
    .filter(F.col("split") == "val")
    .select("userId", "session_id", "executionId", F.col("internalId").alias("trigger_internal_id"))
    .distinct()
)

In [ ]:
val_session_products = (
    val_triggers
    .join(
        df_split.filter(F.col("split") == "val").select("userId", "session_id", "internalId", "emitted"),
        on=["userId", "session_id"],
        how="inner",
    )
    .groupBy("userId", "session_id", "executionId", "trigger_internal_id")
    .agg(
        F.collect_list(
            F.struct(F.col("internalId").alias("internal_id"), F.col("emitted"))
        ).alias("session_products")
    )
)

val_exec2code_pandas = pd.DataFrame(val_exec2code.items(), columns=["executionId", "exec_code"])
df_val_exec2code = spark.createDataFrame(val_exec2code_pandas)

sessions_raw_val_prototype = (
    val_session_products
    .join(df_val_exec2code, on="executionId", how="inner")
    .select(
        "exec_code",
        "trigger_internal_id",
        F.col("executionId").alias("execution_id"),
        "session_id",
        "session_products",
    )
).cache()

# Coverage check: with exec2code built over all executions of the split, the inner join above must
# not drop anything. A gap here means an execution was missing from exec2code.
n_triggers = val_triggers.count()
n_sessions = sessions_raw_val_prototype.count()
print(f"val_triggers: {n_triggers} triggers")
print(f"sessions_raw_val_prototype: {n_sessions} triggers (perdus par le join exec_code: {n_triggers - n_sessions})")
sessions_raw_val_prototype.printSchema()

display(sessions_raw_val_prototype.limit(1))

In [ ]:
sessions_raw_val_prototype.write.mode("overwrite").parquet(
    "s3://mirakl-data-science-tmp2/nbraun/datasets/coview-mdm/sessions_raw_val_prototype.parquet"
)
print("sessions_raw_val_prototype.parquet uploaded")

Création de `prod_results_val_prototype` — les 12 produits renvoyés par la prod pour chaque trigger, en **deux variantes** calculées sur le même tableau `relevantProducts` :

- `_display` : les 12 premiers dans l'ordre du tableau, c'est-à-dire l'ordre d'affichage
- `_relevance` : les 12 premiers par `relevanceScore` décroissant

Les positifs et négatifs sont dérivés par appartenance aux produits de la session. **Aucun filtrage sur les edges du graphe d'entraînement** ici : on veut le vrai retour prod, pas des négatifs d'entraînement.

In [ ]:
# Tie-break sur internalId pour que le tri par relevanceScore soit deterministe en cas d'egalite.
# Ecrit en SQL plutot qu'avec la lambda F.array_sort (qui exige PySpark >= 3.4).
SORT_BY_RELEVANCE = """
array_sort(
    candidates,
    (a, b) -> CASE
        WHEN a.relevanceScore < b.relevanceScore THEN 1
        WHEN a.relevanceScore > b.relevanceScore THEN -1
        WHEN a.internalId > b.internalId THEN 1
        WHEN a.internalId < b.internalId THEN -1
        ELSE 0
    END
)
"""

val_placements_prototype = (
    df_recommended_products_t2s_with_sessions_indexes
    .filter(F.col("split") == "val")
    .select(
        F.col("executionId"),
        F.col("internalId").alias("trigger_internal_id"),
        F.posexplode("sponsoredProductPlacementExecutions").alias("placement_pos", "placement"),
    )
    .select(
        "executionId",
        "trigger_internal_id",
        "placement_pos",
        F.col("placement.relevantProducts").alias("candidates"),
    )
    # one adlog row can be duplicated by the session join; a placement is identified by (exec, pos)
    .dropDuplicates(["executionId", "placement_pos"])
    .withColumn("cands_display", F.slice(F.col("candidates"), 1, 12))
    .withColumn("cands_relevance", F.slice(F.expr(SORT_BY_RELEVANCE), 1, 12))
).cache()

print(f"val_placements_prototype: {val_placements_prototype.count()} placements")
val_placements_prototype.select("executionId", "placement_pos", F.size("candidates").alias("n_candidates")).show(5)

In [ ]:
TOP_K = 12


def prod_products_by_trigger(df_placements: DataFrame, arr_col: str, top_k: int = TOP_K) -> DataFrame:
    """Fusionne les candidats d'un trigger à travers ses placements et garde les top_k premiers.

    Args:
        df_placements: Placements d'un trigger, une ligne par (executionId,
            placement_pos), avec une colonne de candidats déjà triée
            (arr_col).
        arr_col: Nom de la colonne tableau à utiliser (ex. cands_display,
            cands_relevance).
        top_k: Nombre maximal de candidats conservés par trigger après
            fusion.

    Returns:
        DataFrame avec les colonnes executionId, trigger_internal_id,
        products_returned (liste de structs internal_id/rank). Un candidat
        présent dans plusieurs placements du même trigger est fusionné à
        son meilleur rang (pos_rank minimal) ; le rang final réordonne ces
        candidats fusionnés (tie-break sur internal_id) et ne garde que les
        top_k premiers.
    """
    ranked = (
        df_placements
        .select(
            "executionId",
            "trigger_internal_id",
            F.posexplode(arr_col).alias("candidate_pos", "candidate"),
        )
        .select(
            "executionId",
            "trigger_internal_id",
            F.col("candidate.internalId").cast("bigint").alias("internal_id"),
            (F.col("candidate_pos") + F.lit(1)).alias("pos_rank"),
        )
        .groupBy("executionId", "trigger_internal_id", "internal_id")
        .agg(F.min("pos_rank").alias("pos_rank"))
        .withColumn(
            "rank",
            F.row_number().over(
                Window.partitionBy("executionId", "trigger_internal_id")
                      .orderBy(F.col("pos_rank").asc(), F.col("internal_id").asc())
            ),
        )
        .filter(F.col("rank") <= top_k)
    )

    return (
        ranked
        .groupBy("executionId", "trigger_internal_id")
        .agg(
            F.sort_array(
                F.collect_list(F.struct(F.col("rank"), F.col("internal_id")))
            ).alias("ranked")
        )
        .withColumn(
            "products_returned",
            F.transform(
                "ranked",
                lambda r: F.struct(
                    r["internal_id"].alias("internal_id"),
                    r["rank"].alias("rank"),
                ),
            ),
        )
        .select("executionId", "trigger_internal_id", "products_returned")
    )


display_products_prototype = prod_products_by_trigger(val_placements_prototype, "cands_display")
relevance_products_prototype = prod_products_by_trigger(val_placements_prototype, "cands_relevance")

print(f"display: {display_products_prototype.count()} triggers")
print(f"relevance: {relevance_products_prototype.count()} triggers")

In [ ]:
EMPTY_PRODUCTS = F.array().cast("array<struct<internal_id:bigint,rank:int>>")

val_session_ids_prototype = (
    sessions_raw_val_prototype
    .select(
        "exec_code",
        "trigger_internal_id",
        F.col("execution_id").alias("executionId"),
        F.col("session_products.internal_id").alias("session_ids"),
    )
    .dropDuplicates(["exec_code"])
)


def add_pos_neg(df: DataFrame, suffix: str) -> DataFrame:
    """Dérive les listes de positifs et négatifs à partir des produits renvoyés par la production.

    Args:
        df: DataFrame avec les colonnes products_returned_{suffix}
            (candidats renvoyés par la production) et session_ids
            (produits réellement vus dans la session).
        suffix: Variante à traiter ("display" ou "relevance").

    Returns:
        df enrichi de positives_{suffix} et negatives_{suffix} :
        sous-listes de products_returned_{suffix} selon que le produit
        appartient ou non à session_ids. Aucune ligne du DataFrame n'est
        retirée ; contrairement aux négatifs d'entraînement, ces négatifs
        ne sont pas filtrés sur les arêtes du graphe.
    """
    products = F.col(f"products_returned_{suffix}")
    return (
        df
        .withColumn(
            f"positives_{suffix}",
            F.filter(products, lambda p: F.array_contains(F.col("session_ids"), p["internal_id"])),
        )
        .withColumn(
            f"negatives_{suffix}",
            F.filter(products, lambda p: ~F.array_contains(F.col("session_ids"), p["internal_id"])),
        )
    )


prod_results_val_prototype = (
    val_session_ids_prototype
    .join(
        display_products_prototype.withColumnRenamed("products_returned", "products_returned_display"),
        on=["executionId", "trigger_internal_id"],
        how="left",
    )
    .join(
        relevance_products_prototype.withColumnRenamed("products_returned", "products_returned_relevance"),
        on=["executionId", "trigger_internal_id"],
        how="left",
    )
    .withColumn("session_ids", F.coalesce(F.col("session_ids"), F.array().cast("array<bigint>")))
    .withColumn("products_returned_display", F.coalesce(F.col("products_returned_display"), EMPTY_PRODUCTS))
    .withColumn("products_returned_relevance", F.coalesce(F.col("products_returned_relevance"), EMPTY_PRODUCTS))
)

prod_results_val_prototype = add_pos_neg(prod_results_val_prototype, "display")
prod_results_val_prototype = add_pos_neg(prod_results_val_prototype, "relevance")

prod_results_val_prototype = prod_results_val_prototype.select(
    "exec_code",
    "trigger_internal_id",
    F.col("executionId").alias("execution_id"),
    "products_returned_display",
    "positives_display",
    "negatives_display",
    "products_returned_relevance",
    "positives_relevance",
    "negatives_relevance",
).cache()

print(f"prod_results_val_prototype: {prod_results_val_prototype.count()} triggers")
prod_results_val_prototype.printSchema()

display(prod_results_val_prototype.limit(1))

In [ ]:
prod_results_val_prototype.write.mode("overwrite").parquet(
    "s3://mirakl-data-science-tmp2/nbraun/datasets/coview-mdm/prod_results_val_prototype.parquet"
)
print("prod_results_val_prototype.parquet uploaded")

Diagnostic — est-ce que les deux variantes renvoient réellement des ensembles différents, et laquelle fait remonter les produits vus le plus tôt ? Si `pct_identical_sets` est proche de 1, les deux variantes sont équivalentes et la comparaison en aval n'a pas d'objet.

In [ ]:
ids = lambda col: F.transform(col, lambda p: p["internal_id"])

overlap_stats = (
    prod_results_val_prototype
    .select(
        F.size("products_returned_display").alias("n_display"),
        F.size("products_returned_relevance").alias("n_relevance"),
        F.size(F.array_intersect(ids("products_returned_display"), ids("products_returned_relevance"))).alias("overlap"),
        F.size("positives_display").alias("n_pos_display"),
        F.size("positives_relevance").alias("n_pos_relevance"),
    )
    .withColumn(
        "identical_sets",
        (F.col("overlap") == F.col("n_display")) & (F.col("n_display") == F.col("n_relevance")),
    )
)

overlap_stats.agg(
    F.count("*").alias("total_triggers"),
    F.avg("n_display").alias("avg_n_display"),
    F.avg("overlap").alias("avg_overlap"),
    F.avg(F.col("identical_sets").cast("int")).alias("pct_identical_sets"),
    F.avg("n_pos_display").alias("avg_pos_display"),
    F.avg("n_pos_relevance").alias("avg_pos_relevance"),
    F.avg((F.col("n_pos_display") > 0).cast("int")).alias("pct_found_display"),
    F.avg((F.col("n_pos_relevance") > 0).cast("int")).alias("pct_found_relevance"),
).show(truncate=False)


def production_mrr_prototype(df: DataFrame, positives_col: str) -> None:
    """MRR de la prod sur la population COMPLETE de triggers.

    matched-only  : moyenne sur les triggers ayant au moins un positif (comparable a MRR_trigger)
    coverage      : les triggers sans positif comptent 0, denominateur = tous les triggers
    """
    row = (
        df
        .select(F.array_min(F.transform(positives_col, lambda p: p["rank"])).alias("best_rank"))
        .agg(
            F.count("*").alias("total"),
            F.count("best_rank").alias("matched"),
            F.sum(F.lit(1.0) / F.col("best_rank")).alias("sum_rr"),
        )
        .collect()[0]
    )
    total, matched = row["total"], row["matched"]
    sum_rr = row["sum_rr"] or 0.0
    print(
        f"{positives_col:22s} MRR matched-only={sum_rr / matched if matched else 0:.4f} | "
        f"MRR coverage-adjusted={sum_rr / total if total else 0:.4f} | "
        f"{matched}/{total} triggers avec au moins un positif ({matched / total:.2%})"
    )


production_mrr_prototype(prod_results_val_prototype, "positives_display")
production_mrr_prototype(prod_results_val_prototype, "positives_relevance")

In [ ]:
from torch_geometric.data import Data


def build_pyg_data(
    num_nodes: int,
    train_edge_index: torch.Tensor,
    train_neg_edge_index: torch.Tensor,
    train_pos_edge_index: torch.Tensor,
    train_edge_attr: torch.Tensor,
    val_pos_edge_index: torch.Tensor,
    val_neg_edge_index: torch.Tensor,
    test_pos_edge_index: torch.Tensor,
    test_neg_edge_index: torch.Tensor,
    x: torch.Tensor = None,
) -> Data:
    """
    Assemble all tensors into a single PyG Data object.
    """
    data = Data(
        edge_index=train_edge_index,
        edge_attr=train_edge_attr,
        num_nodes=num_nodes,
    )

    if x is not None:
        data.x = x

    data.train_neg_edge_index = train_neg_edge_index
    data.train_pos_edge_index = train_pos_edge_index

    # Evaluation edges
    data.val_pos_edge_index = val_pos_edge_index
    data.val_neg_edge_index = val_neg_edge_index
    data.test_pos_edge_index = test_pos_edge_index
    data.test_neg_edge_index = test_neg_edge_index

    return data

In [ ]:
num_nodes = len(node2idx)

data = build_pyg_data(
    num_nodes=num_nodes,
    train_edge_index=train_edge_index,
    train_neg_edge_index = train_neg_edge_index,
    train_pos_edge_index = train_pos_edge_index,
    train_edge_attr=train_edge_attr,
    val_pos_edge_index=val_pos_edge_index,
    val_neg_edge_index=val_neg_edge_index,
    test_pos_edge_index=test_pos_edge_index,
    test_neg_edge_index=test_neg_edge_index,
    x=x,
)

print(data)

In [ ]:
import os
import json
from torch_geometric.data import InMemoryDataset


class CoViewDataset(InMemoryDataset):
    """
    PyG InMemoryDataset for the co-view graph.
    Saves the Data object + node mapping to disk for fast reloading.
    """
    def __init__(self, root: str, data_obj: Data = None, node2idx: dict = None, transform=None, force_reload: bool = False):
        self._data_obj = data_obj
        self._node2idx = node2idx
        super().__init__(root, transform, force_reload=force_reload)
        self.load(self.processed_paths[0])

    @property
    def processed_file_names(self):
        return ["data.pt"]

    def process(self):
        self.save([self._data_obj], self.processed_paths[0])

        if self._node2idx is not None:
            mapping_path = os.path.join(self.processed_dir, "node2idx.json")
            with open(mapping_path, "w") as f:
                json.dump({int(k): int(v) for k, v in self._node2idx.items()}, f)

In [ ]:
dataset = CoViewDataset(
    root="/dbfs/tmp/nbraun/datasets/coview-mdm-prototype",
    data_obj=data,
    node2idx=node2idx,
    force_reload=True,
)

print(f"Dataset saved to /dbfs/tmp/nbraun/datasets/coview-mdm-prototype/")
print(dataset[0])

In [ ]:
import boto3

# Artefacts suffixes _prototype: exec2code couvre ici toutes les executions du split, donc les
# exec_code diffèrent du pipeline principal. Ecraser les chemins partages casserait model_test /
# model_stat existants, qui lisent des parquets encodes avec l'ancien exec2code.
s3 = boto3.client("s3")

s3.upload_file(
    "/dbfs/tmp/nbraun/datasets/coview-mdm-prototype/processed/data.pt",
    "mirakl-data-science-tmp2",
    "nbraun/datasets/coview-mdm/data_prototype.pt"
)
print("data_prototype.pt uploaded")

s3.upload_file(
    "/dbfs/tmp/nbraun/datasets/coview-mdm-prototype/processed/node2idx.json",
    "mirakl-data-science-tmp2",
    "nbraun/datasets/coview-mdm/node2idx_prototype.json"
)
print("node2idx_prototype.json uploaded")

In [ ]:
exec_mappings = {
    "train": {
        "exec2code": {str(eid): code for eid, code in train_exec2code.items()},
        "code2exec": train_code2exec,
    },
    "val": {
        "exec2code": {str(eid): code for eid, code in val_exec2code.items()},
        "code2exec": val_code2exec,
    },
    "test": {
        "exec2code": {str(eid): code for eid, code in test_exec2code.items()},
        "code2exec": test_code2exec,
    },
}

exec_mappings_path = "/dbfs/tmp/nbraun/datasets/coview-mdm-prototype/processed/exec_mappings.json"
with open(exec_mappings_path, "w") as f:
    json.dump(exec_mappings, f)

s3.upload_file(
    exec_mappings_path,
    "mirakl-data-science-tmp2",
    "nbraun/datasets/coview-mdm/exec_mappings_prototype.json"
)
print("exec_mappings_prototype.json uploaded")